In [2]:
import pandas as pd

fondos_mapeados = pd.read_csv('data/funds_prices.csv')

fondos_mapeados

,Dates,0JKT LN Equity,AACCHIA CI Equity,AAXJ US EQUITY,ABCAI2A LX EQUITY,ABCHNSA LX EQUITY,ABEEMI2 LX EQUITY,ABSDHS1 LX EQUITY,AECHAUC LX EQUITY,AEEVI2E LX EQUITY,...,CH0013841017,CH0025751329,CH0038863350,CH0102484968,CH0126673539,CH0126881561,CH0418792922,CH0432492467,CH1256740924,CH1425684714
0,2006-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2006-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2006-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2006-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2006-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5132,2025-09-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,786.0179,124.0415,124.5634,105.5970,88.2833,329.1560,261.3572,80.0270,138.2600,85.078981
5133,2025-09-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,779.1474,124.8916,125.1013,105.5970,88.2833,329.1560,266.1993,81.3048,138.2600,85.078981
5134,2025-09-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,786.4541,126.6383,125.3603,105.5970,88.2833,329.1560,270.1544,81.1020,155.1391,85.078981
5135,2025-09-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,779.9108,125.5204,121.7106,107.3358,88.2833,320.7737,266.3671,80.0067,155.1391,85.078981


In [3]:
diccionario = pd.read_csv('data/funds_dictionary.csv')
diccionario

,Ticker,Asset Class,Subclass,Geografia,Indice,Sector
0,0JKT LN EQUITY,Equities,Equities,Asia ex-Japan,ISHARES TRUST ISHARES MSCI INDIA ETF,India
1,AAXJ US EQUITY,Equities,Equities,Asia ex-Japan,ISHARES MSCI ALL COUNTRY ASIA ES Japan,General
2,ABCHNSA LX EQUITY,Equities,Equities,Asia ex-Japan,AB SICAV I CHINA A SHARES EQUITY PORTFOLIO,China
3,BGFPABA LN EQUITY,Equities,Equities,Asia ex-Japan,BAILLIE GIFFORD OVERSEAS GROWTH FUNDS ICVC BAI...,General
4,CPXJ LN EQUITY,Equities,Equities,Asia ex-Japan,ISHARES MSCI PACIFIC EX Japan UCITS ETF,General
...,...,...,...,...,...,...
330,CH0012032048,Stock,Stock,Swiss,Roche Hldg DR,Stock
331,CH1256740924,Stock,Stock,Swiss,SGS Rg,Stock
332,CH0418792922,Stock,Stock,Swiss,Sika Rg,Stock
333,CH0126881561,Stock,Stock,Swiss,Swiss Re N,Stock


In [7]:
import re
import pandas as pd

# --- 1) Cargar data ---
precios = pd.read_csv('data/funds_prices.csv')
dicc   = pd.read_excel('/Users/matias/Desktop/Proyectos/ranking-fondos/dict_temp_full_portfolio.xlsx', sheet_name='Hoja1')

# Garantiza que la columna de fechas se llame 'Dates' y sea datetime
if 'Dates' in precios.columns:
    precios['Dates'] = pd.to_datetime(precios['Dates'])
else:
    # Si tu CSV trae otro nombre, cámbialo aquí
    raise ValueError("No se encuentra columna 'Dates' en funds_prices.csv")

# --- 2) Normalizador de tickers (equivale 'Equity'/'EQUITY', colapsa espacios, mayúsculas) ---
def norm_ticker(x: str) -> str:
    if pd.isna(x):
        return x
    s = str(x).strip()
    s = re.sub(r'\s+', ' ', s)            # colapsa múltiples espacios
    s = s.upper()                          # todo a MAYÚSCULAS
    s = re.sub(r'\s+EQUITY$', ' EQUITY', s)  # sufijo EQUITY estandar
    return s

# --- 3) Estandarizar tickers del diccionario ---
dicc = dicc.copy()
if 'Ticker' not in dicc.columns:
    raise ValueError("El diccionario debe tener columna 'Ticker'")
dicc['Ticker_norm'] = dicc['Ticker'].apply(norm_ticker)

# --- 4) Estandarizar columnas del dataframe de precios (excepto 'Dates') ---
price_cols = [c for c in precios.columns if c != 'Dates']
map_cols = {c: norm_ticker(c) for c in price_cols}
precios_std = precios.rename(columns=map_cols)

# Conjuntos útiles
tickers_precios = set(map_cols.values())
tickers_dicc    = set(dicc['Ticker_norm'])

# --- 5) Detectar y reportar no mapeados (para que los corrijas luego si hace falta) ---
no_mapeados = sorted(tickers_precios - tickers_dicc)
# (Opcional) Guarda un CSV para revisarlos:
pd.DataFrame({'Ticker_no_mapeado': no_mapeados}).to_csv('data/tickers_no_mapeados.csv', index=False)

# --- 6) Seleccionar CHILE ---
# Asumimos que 'Geografia' indica el país/region y que Chile aparece como 'Chile'.
# Además, por si tienes acciones locales por ISIN, los de Chile parten con 'CL'.
dicc['Geografia/Subclass'] = dicc['Geografia/Subclass'].fillna('')

mask_chile = (dicc['Geografia/Subclass'].str.strip().str.casefold() == 'chile') | dicc['Ticker_norm'].str.startswith('CL')
dicc_chile = dicc.loc[mask_chile, ['Ticker_norm']].drop_duplicates()

# Tickers de Chile que SÍ están en el dataframe de precios
tickers_chile = sorted(t for t in dicc_chile['Ticker_norm'] if t in tickers_precios)

# --- 7) Separar dataframes: CHILE vs resto ---
cols_chile = ['Dates'] + tickers_chile
df_chile = precios_std.loc[:, [c for c in cols_chile if c in precios_std.columns]].copy()

tickers_resto = sorted(tickers_precios - set(tickers_chile))
cols_resto = ['Dates'] + tickers_resto
df_resto = precios_std.loc[:, [c for c in cols_resto if c in precios_std.columns]].copy()

# --- 8) (Opcional) Guardar resultados para tu pipeline de moneda a futuro ---
df_chile.to_csv('data/funds_prices_chile.csv', index=False)
df_resto.to_csv('data/funds_prices_exchile.csv', index=False)

# --- 9) Resumen rápido por consola ---
print(f"Tickers totales en precios: {len(tickers_precios)}")
print(f"Tickers mapeados en dicc:   {len(tickers_dicc)}")
print(f"Tickers Chile detectados:   {len(tickers_chile)}")
print(f"No mapeados (revisar):     {len(no_mapeados)} -> data/tickers_no_mapeados.csv")


Tickers totales en precios: 338
Tickers mapeados en dicc:   335
Tickers Chile detectados:   20
No mapeados (revisar):     8 -> data/tickers_no_mapeados.csv


In [8]:
# --- 10) Exportar listas de tickers a Excel (Chile / Internacional) ---
from pathlib import Path

# Mapa inverso para recuperar el nombre original como venía en funds_prices.csv
inv_map_cols = {v: k for k, v in map_cols.items()}

# DataFrames de listas
df_list_chile = pd.DataFrame({
    "Ticker_norm": tickers_chile,
    "Ticker_original": [inv_map_cols.get(t, t) for t in tickers_chile]
})

tickers_internac = sorted(tickers_precios - set(tickers_chile))
df_list_internac = pd.DataFrame({
    "Ticker_norm": tickers_internac,
    "Ticker_original": [inv_map_cols.get(t, t) for t in tickers_internac]
})

# (Opcional) también dejar los no mapeados para revisión
df_no_mapeados = pd.DataFrame({"Ticker_no_mapeado": no_mapeados})

# Ruta de salida
out_xlsx = Path("data") / "listas_tickers_por_region.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    df_list_chile.to_excel(writer, sheet_name="Chile", index=False)
    df_list_internac.to_excel(writer, sheet_name="Internacional", index=False)
    df_no_mapeados.to_excel(writer, sheet_name="No_mapeados", index=False)

print(f"✔️ Archivo creado: {out_xlsx.resolve()}")
print(f"Hojas: Chile ({len(df_list_chile)}), Internacional ({len(df_list_internac)}), No_mapeados ({len(df_no_mapeados)})")


✔️ Archivo creado: /Users/matias/Desktop/Proyectos/ranking-fondos/data/listas_tickers_por_region.xlsx
Hojas: Chile (20), Internacional (318), No_mapeados (8)
